# ADBA - Unsloth fallback (Kaggle/Colab)

Notebook nay dung de fine-tune fallback khi khong train MLX local.

## Muc tieu
- Load `train.jsonl` va `valid.jsonl` (ShareGPT format)
- Fine-tune Qwen2.5-Coder 7B Instruct voi LoRA qua Unsloth
- Luu adapter de evaluate va deploy

In [ ]:
# Cài dependencies (Kaggle/Colab)
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install datasets transformers trl peft accelerate bitsandbytes

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

PROJECT_DIR = Path('/kaggle/working/adba')  # doi path neu can
TRAIN_PATH = PROJECT_DIR / 'data' / 'train.jsonl'
VALID_PATH = PROJECT_DIR / 'data' / 'valid.jsonl'
OUT_DIR = PROJECT_DIR / 'adapters' / 'qwen25-adba-unsloth'

assert TRAIN_PATH.exists(), f'Missing: {TRAIN_PATH}'
assert VALID_PATH.exists(), f'Missing: {VALID_PATH}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('TRAIN:', TRAIN_PATH)
print('VALID:', VALID_PATH)
print('OUT  :', OUT_DIR)

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
valid_rows = load_jsonl(VALID_PATH)

print('train rows:', len(train_rows))
print('valid rows:', len(valid_rows))
print('sample keys:', train_rows[0].keys())

In [ ]:
max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

def to_text(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_ds = Dataset.from_list(train_rows).map(to_text)
valid_ds = Dataset.from_list(valid_rows).map(to_text)

print(train_ds)
print(valid_ds)

In [ ]:
# Hyperparams fallback cho Kaggle/Colab (gan voi spec .cursorrules)
batch_size = 2
gradient_accumulation_steps = 4
max_steps = 1200
learning_rate = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    args=SFTConfig(
        output_dir=str(OUT_DIR),
        max_seq_length=max_seq_length,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        logging_steps=50,
        eval_steps=200,
        save_steps=200,
        max_steps=max_steps,
        warmup_steps=50,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        seed=42,
        report_to='none',
        packing=False,
    ),
)

trainer.train()

In [ ]:
# Save adapter + tokenizer
model.save_pretrained(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print('Saved adapter to', OUT_DIR)

## Ghi chu
- Notebook nay la fallback cho GPU cloud.
- Sau khi train xong, copy adapter ve local de danh gia va dong goi.
- Neu can export GGUF/merge LoRA, thuc hien o buoc deployment rieng.